In [ ]:
# OPTIONAL: Install
# pip install -qU langchain langchain-community langchain-openai duckduckgo-search wikipedia numexpr python-dotenv


## Tutorial: Using LangChain Tools (Search, Calculator) 
We’ll register two tools and call them via a tiny router (no heavy agents yet).


In [ ]:
from getpass import getpass
from langchain_openai import ChatOpenAI

# Enter your OpenRouter API key securely when prompted.
OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

# OpenRouter provides an OpenAI-compatible API.
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

# You can change this to any compatible OpenRouter model.
MODEL = "openai/gpt-4o-mini"

llm = ChatOpenAI(
    model=MODEL,
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL,
    temperature=0,
    seed=42,
)

print("OpenRouter configured successfully.")
print("Model:", MODEL)
from langchain.tools import Tool

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, seed=42)


### Step 1: Define tools
We’ll add a calculator and a web search (DuckDuckGo) with narrow scopes.


In [ ]:
import numexpr as ne
from duckduckgo_search import DDGS


def calc(expression: str) -> str:
    try:
        value = ne.evaluate(expression)
        return str(value.item())
    except Exception as e:
        return f"Error: {e}"


def web_search(query: str, max_results: int = 3) -> str:
    with DDGS() as ddgs:
        results = list(ddgs.text(query, max_results=max_results))
    # Return a compact string for the LLM; in production, pass structured results
    lines = [f"- {r.get('title')}: {r.get('href')}" for r in results]
    return "\n".join(lines)[:1200]


calc_tool = Tool(name="calculator", func=calc, description="Evaluate simple math expressions, return a number as string.")
search_tool = Tool(name="search", func=web_search, description="DuckDuckGo web search, returns top links.")

print(calc_tool.run("(3+4)*2"))
print(search_tool.run("LangChain framework overview"))


### Step 2: A tiny “tool-aware” responder
We’ll route based on input shape. This mirrors agent behavior without the overhead.


In [ ]:
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

explain_prompt = PromptTemplate.from_template(
    "Explain briefly: {question}"
)
explain_chain = LLMChain(llm=llm, prompt=explain_prompt)


def route(query: str) -> str:
    looks_math = any(ch.isdigit() for ch in query) and any(op in query for op in "+-*/()")
    looks_search = any(word in query.lower() for word in ["latest", "current", "news", "what is", "overview"]) and not looks_math
    if looks_math:
        return calc_tool.run(query)
    if looks_search:
        return search_tool.run(query)
    return explain_chain.run({"question": query})

print(route("(12+8)/5"))
print(route("latest on langchain memory"))
print(route("What is a PromptTemplate?"))
